# GP Kernel Visualizations

This notebook illustrates how the choice of **kernel** shapes the functions that a Gaussian Process can express.
For each of three kernels, an animation reveals random draws from the GP prior one by one,
building up a picture of the function space encoded by that kernel.

| Kernel | Key property |
|---|---|
| **Squared Exponential (RBF)** | Infinitely smooth, localised correlations |
| **Linear** | Affine (straight-line) functions |
| **Periodic** | Exactly repeating patterns |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
import seaborn as sns

from data_creation import generate_gp_plots_data
from gp_utils import (
    build_progressive_posterior_frames,
    compute_posterior_ylim_from_frames,
)
from kernels import linear_kernel, periodic_kernel, squared_exponential

In [ ]:
X_plot = np.linspace(-5, 5, 300)
sigma = 2.0
n_draws = 8

sns.set_theme(style="whitegrid", palette="husl")
sns.set_context("notebook", font_scale=1.1)


def make_gp_animation(K, kernel_name, seed=42, interval=900):
    mean = np.zeros(len(X_plot))
    sd = np.sqrt(np.diag(K))
    colors = sns.color_palette("husl", n_draws)

    rng = np.random.RandomState(seed)
    samples = [
        rng.multivariate_normal(mean, K + 1e-10 * np.eye(len(X_plot)))
        for _ in range(n_draws)
    ]

    y_lo = min(np.min(mean - sigma * sd), min(s.min() for s in samples))
    y_hi = max(np.max(mean + sigma * sd), max(s.max() for s in samples))
    pad = 0.1 * (y_hi - y_lo)
    y_lo -= pad
    y_hi += pad

    fig, ax = plt.subplots(figsize=(10, 6))

    def draw_frame(i):
        ax.cla()
        ax.fill_between(
            X_plot,
            mean - sigma * sd,
            mean + sigma * sd,
            alpha=0.35,
            label="95% confidence interval",
        )
        ax.plot(X_plot, mean, lw=2, label="Prior mean")
        for j in range(i):
            ax.plot(X_plot, samples[j], lw=1.8, alpha=0.85,
                    color=colors[j], label=f"Draw {j + 1}")
        draw_word = "Draw" if i == 1 else "Draws"
        ax.set_title(
            f"{kernel_name}: {i} Random {draw_word}",
            fontsize=14,
            fontweight="bold",
        )
        ax.set_xlabel("Input (x)", fontsize=12)
        ax.set_ylabel("Output (y)", fontsize=12)
        ax.set_xlim(X_plot[0], X_plot[-1])
        ax.set_ylim(y_lo, y_hi)
        ax.legend(loc="upper right", frameon=True, shadow=True, fontsize=9)
        ax.grid(True, alpha=0.3)

    anim = animation.FuncAnimation(
        fig,
        draw_frame,
        frames=range(n_draws + 1),
        interval=interval,
        repeat=True,
    )
    plt.tight_layout()
    plt.close(fig)
    return anim

## Squared Exponential Kernel

The SE kernel produces **infinitely smooth** functions.
Correlations decay as a Gaussian function of distance:

$$k(x, x') = \sigma_f^2 \exp\!\left(-\frac{(x - x')^2}{2\ell^2}\right)$$

Parameters used: $\ell = 1.0$, $\sigma_f = 1.0$

In [30]:
K_se = squared_exponential(X_plot, X_plot, lengthscale=1.0, variance=1.0)
anim_se = make_gp_animation(K_se, "Squared Exponential Kernel", seed=42)
HTML(anim_se.to_jshtml())

## Linear Kernel

The linear kernel encodes a prior over **affine functions** — straight lines with random slope and intercept:

$$k(x, x') = \sigma_b^2 + \sigma_v^2 \, x \, x'$$

Parameters used: $\sigma_b = 1.0$ (bias variance), $\sigma_v = 1.0$ (slope variance)

In [31]:
K_lin = linear_kernel(X_plot, X_plot, slope_variance=1.0, bias_variance=1.0)
anim_lin = make_gp_animation(K_lin, "Linear Kernel", seed=1)
HTML(anim_lin.to_jshtml())

## Periodic Kernel

The periodic kernel produces **exactly repeating** functions with period $p$:

$$k(x, x') = \sigma_f^2 \exp\!\left(-\frac{2\sin^2\!\left(\dfrac{\pi|x - x'|}{p}\right)}{\ell^2}\right)$$

Parameters used: $p = 2.0$, $\ell = 1.0$, $\sigma_f = 1.0$

In [32]:
K_per = periodic_kernel(X_plot, X_plot, lengthscale=1.0, variance=1.0, period=2.0)
anim_per = make_gp_animation(K_per, "Periodic Kernel", seed=2)
HTML(anim_per.to_jshtml())

## Posterior Updates — Fitting the Data

The following animations show the GP **posterior** for each kernel as observations are added
one by one. All three kernels are conditioned on the **same dataset** (12 points generated
by `data_creation.py`) so you can directly compare how each kernel's inductive bias
shapes the fitted function.

In [ ]:
X_train, y_train, sn = generate_gp_plots_data()
num_points = 10  # animate adding the first 10 observations


def compute_posterior_ylim(kernel_fn):
    """Return (y_lo, y_hi) from posterior frames for kernel_fn."""
    frames = build_progressive_posterior_frames(
        X_train,
        y_train,
        X_plot,
        kernel_fn,
        sn,
        num_points=num_points,
    )
    return compute_posterior_ylim_from_frames(frames, sn, sigma=sigma)


def make_posterior_animation(kernel_fn, kernel_name, interval=1200, ylim=None):
    frames = build_progressive_posterior_frames(
        X_train,
        y_train,
        X_plot,
        kernel_fn,
        sn,
        num_points=num_points,
    )

    if ylim is None:
        p_ymin, p_ymax = compute_posterior_ylim_from_frames(frames, sn, sigma=sigma)
    else:
        p_ymin, p_ymax = ylim

    ci_color = plt.rcParams["axes.prop_cycle"].by_key()["color"][0]

    fig, ax = plt.subplots(figsize=(10, 6))

    def draw_frame(frame_data):
        n, X_n, y_n, mu_n, sd_n = frame_data
        ax.cla()
        ax.fill_between(
            X_plot,
            mu_n - sigma * sd_n,
            mu_n + sigma * sd_n,
            color=ci_color,
            alpha=0.35,
            label="95% confidence interval",
        )
        ax.plot(X_plot, mu_n, color=ci_color, lw=2.2, label="Posterior mean")
        ax.errorbar(
            X_n, y_n, yerr=sn,
            fmt="o", ms=8, elinewidth=2, capsize=4,
            label="Observed data", color="steelblue",
        )
        point_word = "point" if n == 1 else "points"
        ax.set_title(
            f"{kernel_name}: Posterior After {n} Data {point_word}",
            fontsize=14, fontweight="bold",
        )
        ax.set_xlabel("Input (x)", fontsize=12)
        ax.set_ylabel("Output (y)", fontsize=12)
        ax.set_xlim(X_plot[0], X_plot[-1])
        ax.set_ylim(p_ymin, p_ymax)
        ax.legend(loc="best", frameon=True, shadow=True)
        ax.grid(True, alpha=0.3)

    anim = animation.FuncAnimation(
        fig, draw_frame, frames=frames, interval=interval, repeat=True,
    )
    plt.tight_layout()
    plt.close(fig)
    return anim


print(f"Loaded {len(X_train)} training points, noise sigma_n = {sn}")

Loaded 12 training points, noise σ_n = 0.1


### Squared Exponential Kernel — Posterior

The SE posterior interpolates smoothly through the data and quickly recovers the underlying signal.

In [ ]:
def kernel_se(x1, x2):
    return squared_exponential(x1, x2, lengthscale=1.0, variance=1.0)

# Derive shared y-axis from the SE posterior (best-fitting kernel for this data)
shared_post_ylim = compute_posterior_ylim(kernel_se)
anim_se_post = make_posterior_animation(kernel_se, "Squared Exponential Kernel", ylim=shared_post_ylim)
HTML(anim_se_post.to_jshtml())

### Linear Kernel — Posterior

The linear posterior fits a straight line to the data. More observations tighten the credible interval but the model is constrained to affine functions.

In [ ]:
def kernel_lin(x1, x2):
    return linear_kernel(x1, x2, slope_variance=1.0, bias_variance=1.0)

anim_lin_post = make_posterior_animation(kernel_lin, "Linear Kernel", ylim=shared_post_ylim)
HTML(anim_lin_post.to_jshtml())

### Periodic Kernel — Posterior

The periodic posterior tries to explain the data with a repeating pattern. Notice how it extrapolates periodically outside the observed range.

In [ ]:
def kernel_per(x1, x2):
    return periodic_kernel(x1, x2, lengthscale=1.0, variance=1.0, period=2.0)

anim_per_post = make_posterior_animation(kernel_per, "Periodic Kernel")
HTML(anim_per_post.to_jshtml())